# ECQL: LoRA против базовой модели

Перевод вопроса на русском в запрос на внутреннем языке ECQL.

Ноутбук работает в двух местах:
- **Google Colab, T4** 
- база грузится в 4 битах через bitsandbytes;
- **Mac, Apple Silicon** - bitsandbytes не работает, база грузится в bf16.

Порядок: 
- самопроверка метрик, 
- два прогона базовой модели, 
- обучение адаптера,
- прогон с адаптером, 
- сравнение.

## 1. Окружение

In [1]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q \
        transformers==5.15.1 \
        peft==0.20.0 \
        trl==1.10.0 \
        accelerate==1.14.0 \
        datasets==5.0.1 \
        bitsandbytes

from importlib.metadata import PackageNotFoundError, version

for package in ("torch", "transformers", "peft", "trl", "accelerate", "datasets", "bitsandbytes"):
    try:
        print(f"{package:16} {version(package)}")
    except PackageNotFoundError:
        print(f"{package:16} не установлен")

torch            2.13.0
transformers     5.15.1
peft             0.20.0
trl              1.10.0
accelerate       1.14.0
datasets         5.0.1
bitsandbytes     не установлен


## 2. Код и данные

Код лежит в репозитории: `src/ecql_dataset`. 

В Colab репозиторий клонируется, локально берётся из каталога, где лежит ноутбук.

In [2]:
from pathlib import Path

REPO_URL = "https://github.com/samtakoy/llm-engineer-ecql-and-metrcis.git"

if IN_COLAB:
    if not REPO_URL:
        raise RuntimeError("впишите REPO_URL: без него в Colab неоткуда взять код и данные")
    ROOT = Path("/content") / Path(REPO_URL).stem
    if not ROOT.exists():
        !git clone -q {REPO_URL} {ROOT}
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()

sys.path.insert(0, str(ROOT / "src"))

DATASET = ROOT / "dataset" / "ecql"
print("корень:", ROOT)
print("датасет:", DATASET)

корень: /Users/samtakot/devs/learnings/llm-eng/homework/llm-eng-21-training-metrics
датасет: /Users/samtakot/devs/learnings/llm-eng/homework/llm-eng-21-training-metrics/dataset/ecql


In [3]:
import json

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line]

train = read_jsonl(DATASET / "train.jsonl")
val = read_jsonl(DATASET / "val.jsonl")
test = read_jsonl(DATASET / "test.jsonl")


print(f"train {len(train)}, val {len(val)}, test {len(test)}, "
      f"из них challenge {sum(r['meta']['challenge'] for r in test)}")

print()
print(test[0]["input"])
print(test[0]["output"])

train 176, val 32, test 68, из них challenge 15

Отзывы из Пятигорска и Лермонтова
FETCH [REVIEWS] WHERE @city IS 'Пятигорск' || @city IS 'Лермонтов'


## 3. Устройство

Отсюда берутся размер батча и способ загрузки модели.

In [4]:
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.bfloat16
    LOAD_IN_4BIT = True
    BATCH_SIZE = 4
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    DTYPE = torch.bfloat16
    LOAD_IN_4BIT = False
    BATCH_SIZE = 1
else:
    DEVICE = "cpu"
    DTYPE = torch.float32
    LOAD_IN_4BIT = False
    BATCH_SIZE = 1

print({"устройство": DEVICE, "тип": str(DTYPE), "4 бита": LOAD_IN_4BIT, "батч": BATCH_SIZE})

{'устройство': 'mps', 'тип': 'torch.bfloat16', '4 бита': False, 'батч': 1}


## 4. Самопроверка метрик

Метрике подаются эталоны вместо ответов модели. Идеальный прогон обязан дать единицу по синтаксису, логике и строке и ноль галлюцинаций.

Проверка идёт до запуска модели: сломанная метрика делает бессмысленными все последующие цифры.

In [5]:
from ecql_dataset.notebook.eval.metrics import judge, self_check

print(self_check(records=test))

{'синтаксис': 1.0, 'логика': 1.0, 'сущность': 1.0, 'поля': 1.0, 'операторы': 1.0, 'значения': 1.0, 'суффикс': 1.0, 'галлюцинации': 0.0, 'строка': 1.0, 'ответов': 68}


- ответов 68 — весь тест;
- синтаксис 1.0 — все 68 разобрались по грамматике;
- логика 1.0 и пять частей по 1.0 — каждый совпал с собой;
- строка 1.0 — посимвольно тоже;
- галлюцинации 0.0 — чужого языка и выдуманных полей нет.

### Проверка на испорченных ответах

Самопроверка показала, что метрика не занижает. Теперь посмотрим — что не завышает.

Шесть ответов, каждый сломан по-своему. Логика должна упасть везде, кроме перестановки условий: там порядок другой, а смысл тот же.

In [ ]:
reference = "FETCH [PLACES] WHERE @category IS 'food' && @price_rub BELOW 700 AS LIST"
cases = {
    "переставлены условия": "FETCH [PLACES] WHERE @price_rub BELOW 700 && @category IS 'food' AS LIST",
    "испорчено значение": "FETCH [PLACES] WHERE @category IS 'culture' && @price_rub BELOW 700 AS LIST",
    "перепутан оператор": "FETCH [PLACES] WHERE @category IS 'food' && @price_rub ABOVE 700 AS LIST",
    "забыт суффикс": "FETCH [PLACES] WHERE @category IS 'food' && @price_rub BELOW 700",
    "SQL вместо ECQL": "SELECT * FROM places WHERE category = 'food'",
    "выдуманное поле": "FETCH [PLACES] WHERE @cuisine IS 'food' && @price_rub BELOW 700 AS LIST",
}

for name, prediction in cases.items():
    verdict = judge(prediction=prediction, reference=reference)
    print(f"{name:22} строка={int(verdict.exact)} логика={int(verdict.logic)} "
          f"галлюцинации={int(verdict.hallucination)}  {verdict.reason}")